In [92]:
import numpy as np
import math
import pandas as pd
import networkx as nx
import random
from statistics import mean
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
# 任务一：节点的重要性与传播能力分析

# 数据初始化函数
def data_input():
    """
    n,m=map(int,input().split())
    edges=list()
    for i in range(m):
        u,v=map(int,input().split())
        edges.append((u,v))
    """
    n=10,m=18
    edges=[(0, 1),(1, 2),(1, 3),(2, 4),(2, 5),(3, 6),(3, 7),(4, 8),(4, 9),(5, 9),(6, 8),(7, 9),(0, 5),(1, 7),(2, 6),(3, 9),(5, 7),(6, 8)]
    return n,m,edges

#计算不同中心性数值并进行排序
def task1(n,edges):
    G=nx.Graph()
    G.add_edges_from(edges)

    #计算中心性
    degree_centrality=nx.degree_centrality(G)
    closeness_centrality=nx.closeness_centrality(G)
    betweenness_centrality=nx.betweenness_centrality(G)

    centralities={
        "degree":degree_centrality,
        "closeness":closeness_centrality,
        "betweenness":betweenness_centrality
    }
    
    # 排序输出
    for type,values in centralities.items():
        print(f"{type} centrality:")
        for rank,value in sorted(values.items(),key=lambda x: x[1],reverse=True):
            print(f"{rank}: {value:.4f}")

n,m,edges=data_input()

task1(n,edges)

degree centrality:
1: 0.4444
2: 0.4444
3: 0.4444
5: 0.4444
7: 0.4444
9: 0.4444
4: 0.3333
6: 0.3333
0: 0.2222
8: 0.2222
closeness centrality:
2: 0.6429
3: 0.6429
9: 0.6429
1: 0.6000
5: 0.6000
7: 0.6000
4: 0.5625
6: 0.5625
0: 0.4500
8: 0.4286
betweenness centrality:
2: 0.1991
1: 0.1250
5: 0.1250
3: 0.1204
9: 0.1204
4: 0.1111
6: 0.1111
7: 0.0370
8: 0.0139
0: 0.0093


In [91]:
#任务二：社交网络中的舆情传播与干预策略

#设置重复次数
repeats=100000

#数据初始化函数
def data_input():
    """
    n,m=map(int,input().split())
    for i in range(m):
        u,v=map(int,input().split())
        G[u].append(v)
        G[v].append(u)
    status=list(map(int,input().split()))
    init_active=[]
    for i in range(n):
        if status[i]==1:
            init_active.append(i)
    prob=list(map(double,input().split()))
    """
    G={0:[1, 5],1:[0, 2, 3, 7],2:[1, 4, 5, 6],3:[1, 6, 7, 9],4:[2, 8, 9],5:[0, 2, 9, 7],6:[2, 3, 8],7:[1, 3, 5, 9],8:[4, 6],9:[3, 4, 5, 7]}
    init_active=[0,4,7,8]
    prob=[0.6,0.5,0.7,0.8,0.3,0.4,0.6,0.9,0.5,0.3]
    return G,init_active,prob

#采用独立级联模型
def search(G,init_active,prob):
    active=set(init_active)
    new_active=set(init_active)
    steps=[]
    while new_active:
        steps.append(new_active)
        next_active=set()
        for u in new_active:
            for v in G[u]:
                if v not in active:
                    if random.random()<(prob[v]+prob[u])/2: # 以样例中连接两条边的点的激活概率均值作为边的传播概率
                        next_active.add(v)
                        active.add(v)
        new_active=next_active
    return steps

def task2_1():
    steps=search(G,init_active,prob)
    print("第一问：\n",steps)

G,init_active,prob=data_input()
task2_1()



#比较不同干预策略对舆情传播效果的影响（删边/删点）

#重复实验计算平均传播轮次
def repeat_search(G,init_active,prob,repeats):
    results=0
    for i in range(repeats):
        results+=len(search(G, init_active, prob))
    return results/repeats


def strategy(G, init_active, prob, repeats):

    steps=repeat_search(G, init_active, prob, repeats)
    print("第二问：")
    print(f"original: {steps:.2f}")

    strategies=[]

    # 删边
    for u in G.keys():
        for v in G[u]:
            if v>u:
                i=G[u].index(v)
                j=G[v].index(u)
                G[u].remove(v)
                G[v].remove(u)
                avg_steps=repeat_search(G,init_active,prob,repeats)
                strategies.append(((u, v),avg_steps))
                G[u].insert(i,v)
                G[v].insert(j,u)

    # 删点
    for u in G.keys():
        removed_edges=G[u]
        for v in removed_edges:
            G[v].remove(u)
        G[u]=[]
        avg_steps=repeat_search(G,init_active,prob,repeats)
        strategies.append((u,avg_steps))
        G[u]=removed_edges
        for v in G[u]:
            G[v].append(u)

    # 排序
    strategies.sort(key=lambda x: x[1],reverse=True)

    print("\nrank:")
    for strategy in strategies:
        print(f"{strategy[0]}: {strategy[1]:.2f}")# 从好到坏输出所有策略的结果
    return strategies

def task2_2():
    strategies=strategy(G, init_active, prob, repeats)
    print("\nbest:")
    print(f"{strategies[0][0]}: {strategies[0][1]:.2f}")# 输出最好策略

task2_2()

第一问：
 [{0, 8, 4, 7}, {1, 3, 5}, {9, 2, 6}]
第二问：
original: 2.98

rank:
7: 3.59
(3, 7): 3.39
4: 3.24
(2, 4): 3.23
8: 3.21
(6, 8): 3.21
(5, 7): 3.08
0: 3.08
(1, 7): 3.08
(7, 9): 3.07
(0, 5): 3.03
(0, 1): 3.03
(4, 9): 3.00
(4, 8): 2.97
(1, 3): 2.97
(5, 9): 2.96
(3, 6): 2.94
(1, 2): 2.94
(3, 9): 2.93
(2, 5): 2.92
9: 2.90
1: 2.89
5: 2.89
(2, 6): 2.87
3: 2.83
6: 2.73
2: 2.63

best:
7: 3.59


In [100]:
# 任务三：用户行为预测与网络影响分析

def prepare_data(relations,behaviors):
    
    num_users,num_behavior=behaviors.shape
    features=[]
    targets=[]

    for user in range(num_users):
        neighbors=relations[user]
        users=behaviors[user]
        features.append(np.hstack((neighbors,users)))
        targets.append((users==0).astype(int))

    features=np.array(features)
    targets=np.array(targets)
    return features,targets

def predict(relations,behavior):

    # 准备特征和目标数据
    features,targets=prepare_data(relations,behavior)

    # 初始化存储预测结果的矩阵
    predictions=np.copy(behavior)

    # 对每个内容进行单独预测
    for behavior_id in range(targets.shape[1]):

        y=targets[:,behavior_id]

        # 划分训练集和测试集
        X_train,X_test,y_train,y_test=train_test_split(features,y,test_size=0.3,random_state=42)

        # 训练随机森林分类器
        model=RandomForestClassifier(random_state=42)
        model.fit(X_train,y_train)

        # 对所有用户进行预测
        content_predictions=model.predict(features)

        # 更新未点赞内容的预测结果
        for user_id in range(len(content_predictions)):
            if behavior[user_id,behavior_id]==0:
                predictions[user_id,behavior_id]=content_predictions[user_id]

    return predictions

def data_input():

    # 样例中的邻接矩阵仅有14行15列，故补齐一行
    n=15
    relations=np.array([
        [0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
        [1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0],
        [0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0],
        [1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0],
        [0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1],
        [0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0],
        [0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0],
        [0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0],
        [1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
    ])

    behavior=np.array([
        [1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0],
        [0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0],
        [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0],
        [0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1],
        [1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
        [0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1],
        [1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0],
        [0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0],
        [1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1],
        [0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0],
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0],
        [0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0],
        [1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0]
    ])
    return relations,behavior

def task3():

    relations,behavior=data_input()
    predictions=predict(relations, behavior)
    print(predictions)

task3()


[[1 0 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1 0 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [1 0 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1 0 1 1 1 1]
 [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]]
